# Tutorial 06: Goodness of Fit and Nested Models

#### Lecture and Tutorial Learning Goals:
After completing this week's lecture and tutorial work, you will be able to:

1. List model metrics that are suitable for evaluation of a statistical model developed to make inferences about the data-generating mechanism (e.g., $R^2$, $\text{AIC}$, Likelihood ratio test/$F$-test), their strengths and limitations, as well as how they are calculated.
2. Write a computer script to calculate these model metrics. Interpret and communicate the results from that computer script.
3. Explain how an $F$-test to compare nested models can be used as a variable selection methods.

In [1]:
# Run this cell before continuing.

library(broom)
library(tidyverse)
source("tests_tutorial_06.R")

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘testthat’


The following object is masked from ‘package:dplyr’:

    matches


The following object is masked from ‘package:purrr’:

    is_null


The following objects are masked from ‘package:readr’:

    edition_get, local_edition


The following object is masked from ‘package:tidyr’:

    matches




## Can we predict protein from mRNA?

In *Worksheet 06*, you studied the significance of `mrna` and analyzed the goodness-of-fit of some models. However, there are other models that can be compared. For example, are interaction terms needed, or should we just use an additive model? 

Consider the following models using a dataset with 3 randomly selected genes:

- model.1: $\text{prot}_t=\beta_0 + \varepsilon_t$ 

- model.2:  $\text{prot}_t=\beta_0 + \beta_1 \text{mrna}_{t} + \varepsilon_t$ 

- model.3:  $\text{prot}_t=\beta_0 + \beta_2 \text{gene2}_{t} + \beta_3 \text{gene3}_{t} + \varepsilon_t$ 

- model.4:  $\text{prot}_t=\beta_0 + \beta_1 \text{mrna}_{t} + \beta_2 \text{gene2}_{t} + \beta_3 \text{gene3}_{t} + \varepsilon_t$ 

- model.5:  $\text{prot}_t=\beta_0 + \beta_1 \text{mrna}_{t} + \beta_2 \text{gene2}_{t} + \beta_3 \text{gene3}_{t} + \beta_4 \text{gene2}_{t}\text{mrna}_{t} + \beta_5 \text{gene3}_{t}\text{mrna}_{t} + \varepsilon_t$ 

In [2]:
# Read and take a look at the data.
dat_bio <- read.csv("data/nature_dat.csv", row.names = 1, stringsAsFactors=TRUE)

str(dat_bio)
head(dat_bio,3)
tail(dat_bio,3)

'data.frame':	16704 obs. of  4 variables:
 $ gene  : Factor w/ 1392 levels "ENSG00000000419",..: 1 2 3 4 5 6 7 8 9 10 ...
 $ tissue: Factor w/ 12 levels "adrenal.gland",..: 12 12 12 12 12 12 12 12 12 12 ...
 $ prot  : num  9.97e-06 3.63e-05 1.69e-05 6.75e-05 5.55e-05 ...
 $ mrna  : num  3.44e-05 1.42e-05 1.92e-05 3.64e-05 3.89e-05 ...


,gene,tissue,prot,mrna
,<fct>,<fct>,<dbl>,<dbl>
1,ENSG00000000419,uterus,9.966484e-06,3.44e-05
3,ENSG00000000971,uterus,3.633516e-05,1.42e-05
5,ENSG00000001084,uterus,1.693588e-05,1.92e-05


,gene,tissue,prot,mrna
,<fct>,<fct>,<dbl>,<dbl>
57882,ENSG00000262246,esophagus,3.337902e-05,6.90e-06
57886,ENSG00000269190,esophagus,3.703505e-06,7.80e-06
57888,ENSG00000272325,esophagus,2.574201e-05,1.35e-05


In [3]:
#run this cell
set.seed(561)
dat_3genes <- dat_bio  %>%  
         subset(gene %in% sample(gene,3)) 

**Question 1.0**
<br>{points: 4}

We can use the adjusted $R^2$ to compare the goodness-of-fit of `model.5` and `model.4` to conclude which fits the data better. 

a) [2pts] Use the dataset `dat_3genes` to fit both models and the function `glance()` to obtain their $R^2$ and adjusted $R^2$. 

b) [1pts] Compare the adjusted $R^2$ of the two models and discuss the results. 

c) [1pts] Compare the $R^2$ of the two models and explain why that of `model.5` is larger.

In [4]:
# Your code and numerical results go here. We will grade this cell manually

# your code here

# Part A

model_4 <- lm(prot ~ gene + mrna, data = dat_3genes)
model_5 <- lm(prot ~ gene * mrna, data = dat_3genes)

glance(model_4)
glance(model_5)

r.squared,adj.r.squared,sigma,statistic,p.value,df,logLik,AIC,BIC,deviance,df.residual,nobs
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<int>
0.4142025,0.3592839,5.688132e-05,7.542127,0.0005944789,3,302.9219,-595.8438,-587.9262,1.035355e-07,32,36


r.squared,adj.r.squared,sigma,statistic,p.value,df,logLik,AIC,BIC,deviance,df.residual,nobs
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<int>
0.4405562,0.3473155,5.741013e-05,4.724937,0.002663059,5,303.7504,-593.5009,-582.4162,9.887768e-08,30,36


> **Part B:**
> Despite model 4 having a smaller R^2 value compared to model 5, it actually has a slightly larger adjusted R^2 value compared to model 5. The adjusted R^2 value is the variant of the R^2 value that penalizes the number of predictors used in the model. This is a more suitable value to compare the two models since this allows us to compare the proportion of the total variance in the response that is explained by each of the models while accounting for the differing number of predictors. Therefore, when accounting for the number of predictors, model 4 (the additive model) has a higher proportion of the total variance in the response that is explained by the model (while accounting for number of predictors).
> 
> **Part C:**
> Model 4 (additive model) has a smaller R^2 value compared to model 5 (interaction term model). This means that model 5 has a higher proportion of the total variation in the response that is explained by the model compared to model 4. However, this does not take into account the number of predictors. Since model 5 includes all predictors in model 4 as well as the interaction terms, there are more predictors in the model overall. As more predictors are added to the model, the R^2 value will naturally increase, therefore the adjusted R^2 value is a more appropriate indicator of how well the model explains the variance of the data since it accounts for the number of predictors in the two models. Since model 4 is nested in model 5, adding the additional terms will only shrink the RSS; since R^2 is calculated by the following: `1-RSS/TSS`, a shrinking RSS will result in a direct increase in R^2.

**Question 1.1**
<br>{points: 1}

Use the function `anova()` to test if the model with interaction terms (`model.5`) is significantly different from an additive one (`model.4`). 

> Note that both models have `mrna`,  but the full model, `model.5`, assumes that the expected change in protein levels per unit change in mRNA levels differs for each gene.

Store the output of the `anova` function in an object called `Ftest_3genes_add_full`.

*Write your own code and run it.*

In [5]:
#[write your code here]

# Hints: 
# - fit the additive model
# - fit the full model
# - call the anova function and store the output in Ftest_3genes_add_full


# your code here
Ftest_3genes_add_full <- anova(model_4, model_5)

Ftest_3genes_add_full

,Res.Df,RSS,Df,Sum of Sq,F,Pr(>F)
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,32,1.035355e-07,NA,NA,NA,NA
2,30,9.887768e-08,2,4.657826e-09,0.7066043,0.5013423


In [6]:
test_1.1()

Test passed 🌈
Test passed 🥇
[1] "Success!"


**Question 1.2**
<br>{points: 1}

Using a significance level $\alpha = 0.05$ and the results in `Ftest_3genes_add_full`, in plain words, what is the conclusion from the test run results in *Question 1.1*?

**A.** We reject the null hypothesis; thus, the *full* model is significatly better than the *reduced* model.

**B.** We fail to reject the null hypothesis; thus, there is not enough evidence that the *full* model with additional interaction terms is better than the additive (reduced) model.

**C.** We accept the alternative hypothesis; thus, the *full* model is better than the *reduced* model.

**D.** We do not accept the alternative hypothesis; thus, the *full* model with additional interaction terms is not better than the *reduced* model.

*Assign your answer to an object called `answer1.2`. Your answer should be one of `"A"`, `"B"`, `"C"`, or `"D"` surrounded by quotes.*

In [7]:
# answer1.2 <- 

# your code here
answer1.2 <- "B"

In [8]:
test_1.2()

Test passed 🎉
Test passed 🥇
Test passed 🥳
[1] "You are doing great!"


#### Assessing mRNA in the additive model

As a final test, let's examine the significance of `mrna` in the additive model.

**Question 1.3**
<br>{points: 2}

Compare the additive model with mRNA as an input and distinct intercepts per gene  (`model.4`) with a model without `mrna` and only the categorical variable `gene` as input variable (`model.3`). Note that the second model predicts protein levels with the average protein level within each gene.

Use the function `tidy()` to obtain a summary of the additive model. Include the corresponding asymptotic 90% confidence intervals. Store the results in an object called `add_mrna_results`.

Use the function `anova()` to compare these models and store the results in an object called `Ftest_3genes_add_mrna`.

*Fill out those parts indicated with `...`, uncomment the corresponding code in the cell below, and run it.*

In [9]:
#[write your code here]

#add_mrna_results <- ...
#Ftest_3genes_add_mrna <- ...

# your code here
model_3 <- lm(prot~gene, data=dat_3genes)
add_mrna_results <- tidy(model_4, conf.int=TRUE, conf.level=0.90)
Ftest_3genes_add_mrna <- anova(model_3, model_4)

add_mrna_results
Ftest_3genes_add_mrna

term,estimate,std.error,statistic,p.value,conf.low,conf.high
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
(Intercept),-2.970840e-05,4.091307e-05,-0.7261349,0.4730341541,-9.901059e-05,3.959378e-05
geneENSG00000143553,3.446627e-05,3.617459e-05,0.9527759,0.3478455325,-2.680945e-05,9.574200e-05
geneENSG00000168497,1.220579e-04,3.297606e-05,3.7014098,0.0008038126,6.620014e-05,1.779157e-04
mrna,6.043032e-01,4.144523e-01,1.4580767,0.1545649197,-9.773283e-02,1.306339e+00


,Res.Df,RSS,Df,Sum of Sq,F,Pr(>F)
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,33,1.104141e-07,NA,NA,NA,NA
2,32,1.035355e-07,1,6.8786e-09,2.125988,0.1545649


In [10]:
test_1.3.0()
test_1.3.1()

Test passed 🥇
Test passed 😸
Test passed 🎉
Test passed 🥇
Test passed 😸
Test passed 😸
Test passed 😀
Test passed 🎉
Test passed 😀
Test passed 😀
[1] "Success!"
Test passed 🎊
Test passed 🎊
[1] "Success!"


**Question 1.4**
<br>{points: 2}

Compare the $p$-value for `mrna` in `add_mrna_results` with that reported in `Ftest_3genes_add_mrna`. What do you observe? Indicate the null hypotheses tested in each case and explain the results.

> *The p-value for `mrna` in `add_mrna_results` is `0.155`, therefore at a 0.10 significance level, we fail to reject the null hypothesis that `mrna` is not statistically significant (i.e. at a 0.10 significance level, there is not enough evidence to show that `mrna` is statistically significant). When observing `Ftest_3genes_add_mrna`, we can see that the p-value is the exact same as before (`0.155`). In `Ftest_3genes_add_mrna`, the p-value comes from comparing `model_3` and `model_4`, which differ solely by the addition of the `mrna` variable as a predictor. The p-values are equivalent because the two-sided t-test in `add_mrna_results` is equivalent to the F-test in `Ftest_3genes_add_mrna`.*
>
> Null hypothesis for `add_mrna_results`: mrna is not a statistically significant predictor for the model.
> Null hypothesis for `Ftest_3genes_add_mrna`: the model that includes mrna is not statistically more significant than the nested model that doesn't include the mrna predictor.
>
> As previously mentioned, at a 0.10 significance level, we fail to reject both of these hypotheses.

**Question 1.5**
<br>{points: 1}

Using a **significance level $\alpha = 0.10$** and the results in `add_mrna_results`, which of the following claims is correct? 

**A.** The `model.4` that includes `mRNA` is significantly different from `model.3`.

**B.** There is not enough evidence that the `model.4` that includes `mRNA` as a predictor is significantly better than `model.3`.

**C.** The `model.4` that includes `mrna` as a predictor is equivalent to `model.3` since the coefficient for `mrna` is not significantly different from zero.

**D.** The variable `mRNA` is essential to predict protein levels.

*Assign your answer to an object called `answer1.5`. Your answer should be one of `"A"`, `"B"`, `"C"`, or `"D"` surrounded by quotes.*

In [11]:
# answer1.5 <- 

# your code here
answer1.5 <- "B"

In [12]:
test_1.5()

Test passed 🥇
Test passed 🌈
Test passed 🎊
[1] "Success!"
